# Analisi — CV stratificata ripetuta + Nadeau-Bengio

**Domanda:** aggiungere **sesso + età** migliora il rilevamento del PDAC rispetto al solo
dato-immagine? Qui usiamo **K-fold stratificata ripetuta** (5×10 = 50 fold): stime stabili
con intervallo di confidenza. Il confronto con/senza clinica usa il **t-test corretto di
Nadeau-Bengio** (corregge la non-indipendenza tra fold → la scelta statisticamente giusta
per la CV).

> Questo notebook è **autoconsistente**: tutta la logica è qui dentro, cella per cella.
> Non importa nulla da `panorama_ml`. Kernel: **PANORAMA ML (.venv-ml)**.
> ✅ È l'analisi **consigliata per il pilota** (326 casi).

In [1]:
# --- Setup e RIPRODUCIBILITÀ -------------------------------------------------
# Fissiamo tutti i semi casuali: rieseguendo si ottengono SEMPRE gli stessi numeri.
import os, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)
print("Semi casuali fissati a", SEED)

# Trova la radice del progetto (funziona da qualunque cartella: notebooks/, ml_study/, radice).
# Cerca la cartella che contiene le feature e/o il file clinico -> così l'analisi gira
# anche copiando solo ml_study/ + cache/clinical_information.xlsx (senza imagesTr).
CWD = Path.cwd()
def _e_radice(p):
    return (p / "ml_study" / "cache_features").exists() or (p / "cache" / "clinical_information.xlsx").exists()
PROJECT_DIR = next((p for p in [CWD, *CWD.parents] if _e_radice(p)), CWD)
CLINICAL   = PROJECT_DIR / "cache" / "clinical_information.xlsx"
CACHE_FEAT = PROJECT_DIR / "ml_study" / "cache_features"
print("Radice progetto:", PROJECT_DIR)
print("File clinico presente:", CLINICAL.exists(), "| cartella feature:", CACHE_FEAT.exists())

Semi casuali fissati a 42
Radice progetto: D:\PANORAMA_v2
File clinico presente: True | cartella feature: True


## 1) La coorte: 326 casi
Costruiamo l'elenco dei pazienti dello studio. Regola: teniamo solo i casi **scaricati**
che hanno **sesso ed età** (le due variabili cliniche che vogliamo testare). Questo esclude
automaticamente i sottoinsiemi pubblici NIH e MSD (privi di quei metadati) → restano i due
centri olandesi RUMC + UMCG.

- **`y`** = 1 se PDAC, 0 se controllo
- **`sex_bin`** = 0 femmina, 1 maschio
- **`age`** = età in anni (il file clinico la salva come `'077Y'`, la convertiamo in 77)

In [2]:
# --- Costruzione coorte (INLINE) --------------------------------------------
df = pd.read_excel(CLINICAL).rename(columns={"PANORAMA_study_id": "study_id"})

# età: '077Y' -> 77.0
df["age"] = pd.to_numeric(df["patient_age"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
# sesso -> 0/1
sx = df["patient_sex"].astype(str).str.strip().str.upper().str[0]
df["sex_bin"] = sx.map({"F": 0, "M": 1})
# etichetta PDAC
df["y"] = (df["label"].astype(str).str.upper() == "PDAC").astype(int)

# solo casi con feature già estratte (usiamo il file radiomica come riferimento dei presenti)
present = set(pd.read_parquet(CACHE_FEAT / "feat_radiomics.parquet")["study_id"])
cohort = df[df["study_id"].isin(present) & df["sex_bin"].notna() & df["age"].notna()].copy()
cohort = cohort[["study_id", "y", "age", "sex_bin", "label", "level"]].reset_index(drop=True)

print("Casi nella coorte:", len(cohort))
print("PDAC vs controlli:\n", cohort["y"].map({1: "PDAC", 0: "controllo"}).value_counts().to_string())
print("\nSesso per gruppo:")
print(pd.crosstab(cohort["y"].map({1: "PDAC", 0: "controllo"}), cohort["sex_bin"].map({0: "F", 1: "M"})).to_string())
for g, s in cohort.groupby(cohort["y"].map({1: "PDAC", 0: "controllo"})):
    print(f"  età {g}: mediana {s['age'].median():.0f} (range {s['age'].min():.0f}-{s['age'].max():.0f})")
cohort.head()

Casi nella coorte: 326
PDAC vs controlli:
 y
PDAC         182
controllo    144

Sesso per gruppo:
sex_bin     F   M
y                
PDAC       91  91
controllo  73  71
  età PDAC: mediana 71 (range 41-89)
  età controllo: mediana 64 (range 23-99)


,study_id,y,age,sex_bin,label,level
0,100012_00001,0,73.0,1.0,non-PDAC,radiology
1,100030_00001,1,51.0,0.0,PDAC,histopathology
2,100031_00001,0,38.0,1.0,non-PDAC,radiology
3,100037_00001,0,23.0,1.0,non-PDAC,radiology
4,100043_00001,1,73.0,0.0,PDAC,pathology


## 2) Le feature-immagine (già estratte, in cache)
Per ogni paziente abbiamo un vettore di numeri che riassume la sua TC, prodotto da diversi
**estrattori**. Sono già stati calcolati e salvati in `cache_features/*.parquet` (vedi il
notebook `01_come_si_estraggono_le_features`). Qui li **carichiamo e basta** — sono dati puri.

| Estrattore | Cos'è |
|---|---|
| `radiomics` | feature pyradiomics (forma + intensità + texture) dalla ROI |
| `merlin` | embedding di un foundation model di **CT addominale** (Stanford) |
| `ctfm`, `medicalnet`, `models_genesis`, `stu_net` | embedding di foundation model 3D |
| `imagenet`, `radimagenet` | embedding di backbone 2D (foto naturali / radiologia) |

In [3]:
# --- Carica le feature dai parquet (INLINE) ---------------------------------
NOMI = ["radiomics", "merlin", "ctfm", "medicalnet", "models_genesis",
        "stu_net", "imagenet", "radimagenet"]
features = {}
for nome in NOMI:
    p = CACHE_FEAT / f"feat_{nome}.parquet"
    if p.exists():
        features[nome] = pd.read_parquet(p)
        print(f"  {nome:16s}: {features[nome].shape[1]-1:4d} feature, {len(features[nome])} casi")
    else:
        print(f"  {nome:16s}: MANCANTE (verrà saltato)")
print("\nEstrattori disponibili:", list(features))

  radiomics       :  107 feature, 326 casi
  merlin          : 2048 feature, 326 casi
  ctfm            :  512 feature, 326 casi
  medicalnet      : 2048 feature, 326 casi
  models_genesis  :  512 feature, 326 casi
  stu_net         :  512 feature, 326 casi


  imagenet        : 2048 feature, 326 casi
  radimagenet     : 2048 feature, 326 casi

Estrattori disponibili: ['radiomics', 'merlin', 'ctfm', 'medicalnet', 'models_genesis', 'stu_net', 'imagenet', 'radimagenet']


## 3) Gli strumenti (definiti qui, così li vedi)
Definiamo — **inline** — le funzioni che useremo:
- **`build_Xy`**: allinea le feature alla coorte e prepara le matrici (immagine, clinica, y).
- **`make_model`**: il classificatore (regressione logistica L2, oppure random forest).
- **`run_experiment`**: dato uno split train/test, addestra e valuta TRE modelli —
  *solo-immagine*, *immagine+sesso+età*, *solo-clinica* (pavimento) — e restituisce le
  probabilità predette sul test.
- **`make_strata`**: chiave di stratificazione (malattia × sesso × fascia d'età) per split bilanciati.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def build_Xy(feat_df, cohort):
    """Allinea per study_id; ritorna X_img, clinica[eta',sesso], y, e la coorte allineata."""
    m = cohort[["study_id", "y", "age", "sex_bin"]].merge(feat_df, on="study_id", how="inner")
    drop = {"study_id", "y", "age", "sex_bin", "roi_used_mask"}
    img_cols = [c for c in m.columns if c not in drop]
    X_img = m[img_cols].to_numpy(float)
    clin  = m[["age", "sex_bin"]].to_numpy(float)
    y     = m["y"].to_numpy(int)
    return X_img, clin, y, m[["study_id", "y", "age", "sex_bin"]].reset_index(drop=True)

def make_model(kind):
    """Impute (NaN->mediana) + standardizza + classificatore. Deterministico."""
    if kind == "logreg":
        clf = LogisticRegression(penalty="l2", C=0.1, max_iter=5000, class_weight="balanced")
    else:  # "rf"
        clf = RandomForestClassifier(n_estimators=400, max_features="sqrt", min_samples_leaf=3,
                                     class_weight="balanced", random_state=SEED, n_jobs=-1)
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()), ("clf", clf)])

def run_experiment(X_img, clin, y, tr, te, kind="logreg"):
    """Addestra su tr, predice su te i 3 modelli. Le feature-immagine sono IDENTICHE
       nei bracci 'senza' e 'con': così isoliamo l'effetto della clinica."""
    m0 = make_model(kind).fit(X_img[tr], y[tr]);            p0 = m0.predict_proba(X_img[te])[:, 1]
    Xc_tr = np.hstack([X_img[tr], clin[tr]]); Xc_te = np.hstack([X_img[te], clin[te]])
    m1 = make_model(kind).fit(Xc_tr, y[tr]);               p1 = m1.predict_proba(Xc_te)[:, 1]
    m2 = make_model("logreg").fit(clin[tr], y[tr]);        p2 = m2.predict_proba(clin[te])[:, 1]
    return dict(y_test=y[te], p_senza=p0, p_con=p1, p_clinica=p2)

def make_strata(sub):
    """malattia × sesso × fascia d'età (terzili), per split stratificati."""
    ab = pd.qcut(sub["age"], 3, labels=False, duplicates="drop").astype("Int64").astype(str)
    return (sub["y"].astype(str) + "_" + sub["sex_bin"].astype("Int64").astype(str) + "_" + ab).to_numpy()

def auc(y_true, p):
    try: return roc_auc_score(y_true, p)
    except Exception: return np.nan

print("Funzioni pronte: build_Xy, make_model, run_experiment, make_strata, auc")

Funzioni pronte: build_Xy, make_model, run_experiment, make_strata, auc


In [5]:
# Quali modelli valutiamo: radiomica con DUE classificatori (logreg + RF), i deep con logreg.
def righe_modelli():
    for nome in features:
        if nome == "radiomics":
            yield (f"{nome} · logreg", nome, "logreg")
            yield (f"{nome} · random_forest", nome, "rf")
        else:
            yield (f"{nome} · logreg (linear probe)", nome, "logreg")

## 4) Il test statistico: Nadeau-Bengio (per la CV)

In [6]:
from scipy import stats as sps
def nadeau_bengio(deltas, n_train, n_test):
    """t-test corretto di Nadeau-Bengio per la CV ripetuta. I Δ per-fold NON sono
       indipendenti (i training set si sovrappongono): questa correzione gonfia la
       varianza di conseguenza, evitando p-value troppo ottimistici.
       Ritorna (media, ci_low, ci_high, p)."""
    d = np.asarray(deltas, float); J = len(d)
    mean = d.mean(); var = d.var(ddof=1)
    if var == 0 or J < 2:
        return float(mean), float(mean), float(mean), 1.0
    rho = n_test / float(n_train)
    se = np.sqrt(var * (1.0 / J + rho))          # varianza corretta
    t = mean / se; p = 2 * sps.t.sf(abs(t), J - 1)
    tc = sps.t.ppf(0.975, J - 1)
    return float(mean), float(mean - tc * se), float(mean + tc * se), float(p)
print("Funzione nadeau_bengio pronta.")

Funzione nadeau_bengio pronta.


## 5) Eseguiamo e leggiamo i risultati
Per ogni modello: 50 fold; raccogliamo AUROC per-fold (solo-img e +clinica) e i **Δ per-fold**,
poi media, intervallo di confidenza e **p di Nadeau-Bengio**.

In [7]:
from sklearn.model_selection import RepeatedStratifiedKFold
righe = []
for label, nome, kind in righe_modelli():
    Xi, cl, y, sub = build_Xy(features[nome], cohort)
    strata = make_strata(sub)
    a0s, a1s, deltas, ntr, nte = [], [], [], [], []
    rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)
    for tr, te in rkf.split(np.arange(len(y)), strata):
        r = run_experiment(Xi, cl, y, tr, te, kind)
        a0 = auc(r["y_test"], r["p_senza"]); a1 = auc(r["y_test"], r["p_con"])
        a0s.append(a0); a1s.append(a1); deltas.append(a1 - a0); ntr.append(len(tr)); nte.append(len(te))
    a0s, a1s = np.array(a0s), np.array(a1s)
    mean, lo, hi, p = nadeau_bengio(deltas, int(np.mean(ntr)), int(np.mean(nte)))
    righe.append(dict(modello=label,
                      AUROC_solo_img=round(np.nanmean(a0s), 3),
                      img_CI=f"[{np.nanpercentile(a0s,2.5):.2f},{np.nanpercentile(a0s,97.5):.2f}]",
                      AUROC_img_clinica=round(np.nanmean(a1s), 3),
                      delta_medio=round(mean, 3), delta_CI=f"[{lo:.3f},{hi:.3f}]",
                      p_NadeauBengio=round(p, 3), n_folds=len(deltas)))
tab = pd.DataFrame(righe).sort_values("AUROC_solo_img", ascending=False).reset_index(drop=True)
tab

,modello,AUROC_solo_img,img_CI,AUROC_img_clinica,delta_medio,delta_CI,p_NadeauBengio,n_folds
0,merlin · logreg (linear probe),0.905,"[0.83,0.96]",0.905,0.000,"[-0.001,0.001]",0.888,50
1,radiomics · logreg,0.886,"[0.77,0.97]",0.894,0.008,"[-0.003,0.020]",0.159,50
2,radiomics · random_forest,0.814,"[0.70,0.90]",0.819,0.006,"[-0.006,0.017]",0.328,50
3,stu_net · logreg (linear probe),0.784,"[0.69,0.86]",0.784,-0.000,"[-0.003,0.003]",0.954,50
4,imagenet · logreg (linear probe),0.778,"[0.68,0.84]",0.780,0.003,"[-0.001,0.006]",0.132,50
5,medicalnet · logreg (linear probe),0.757,"[0.67,0.82]",0.762,0.005,"[-0.002,0.012]",0.142,50
6,ctfm · logreg (linear probe),0.756,"[0.67,0.84]",0.767,0.010,"[0.000,0.020]",0.044,50
7,models_genesis · logreg (linear probe),0.736,"[0.64,0.84]",0.740,0.004,"[-0.002,0.011]",0.198,50
8,radimagenet · logreg (linear probe),0.735,"[0.65,0.82]",0.739,0.004,"[-0.001,0.008]",0.141,50


In [8]:
sig = tab[tab["p_NadeauBengio"] < 0.05]
print("Miglioramenti SIGNIFICATIVI (Nadeau-Bengio p<0.05):", list(sig["modello"]) if len(sig) else "nessuno")
print("Direzione del Δ (righe > 0):", int((tab["delta_medio"] > 0).sum()), "su", len(tab))

Miglioramenti SIGNIFICATIVI (Nadeau-Bengio p<0.05): ['ctfm · logreg (linear probe)']
Direzione del Δ (righe > 0): 7 su 9


## 6) Come interpretare (onestà)
- Confronta **`AUROC_solo_img`** con **`AUROC_img_clinica`**: se la seconda è più alta la clinica
  *sembra* aiutare; **`p`** < 0.05 → differenza statisticamente significativa.
- **`AUROC_solo_clinica`** = il *pavimento* (quanto segnale c'è in sesso+età da soli).
- **Aspettativa onesta sul pilota (326 casi):** con un effetto piccolo e pochi casi, è normale
  ottenere risultati **non significativi**. Guarda la **coerenza della direzione** tra estrattori:
  è il segnale più affidabile. La risposta statistica vera arriverà sul **dataset completo (2238)**.